# RAG Pipeline — PGABL Generative AI

**Nama**    : Antonius Vincent Jung  
**Model**   : `antonius-vincent/Qwen2.5-3B-Instruct-Indonesian-QLoRA`  
**Teknik**  : RAG (Retrieval-Augmented Generation) dengan Ensemble Retriever  
**Dataset** : Empat Undang-Undang Indonesia (PDF)  

Pipeline:
```
PDF → Extract → Clean → Parent/Child Chunks → Embed (bge-m3)
    → FAISS + BM25 → Ensemble → Parent-Child → LLM → Gradio
```

## Section 1 — Instalasi Dependency

In [1]:
%%capture
# Framework & model
!pip install --upgrade pip -q
!pip install unsloth -q

# RAG stack
!pip install langchain-community langchain-core langchain-huggingface langchain-classic -q
!pip install faiss-cpu -q
!pip install rank_bm25 -q
!pip install PyMuPDF -q
!pip install sentence-transformers -q

# Interface
!pip install gradio -q
!pip install huggingface_hub -q

print('Instalasi selesai.')


## Section 2 — Import Library

In [2]:
import re, gc, uuid
import torch
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import fitz                          # PyMuPDF
from sentence_transformers import SentenceTransformer
import faiss
from rank_bm25 import BM25Okapi

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS as LangFAISS
from langchain_huggingface import HuggingFaceEmbeddings   # ganti dari langchain_community (deprecated)
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever  # pindah dari langchain ke langchain-classic di v1.0+

from huggingface_hub import login as hf_login
from unsloth import FastLanguageModel
import gradio as gr
from IPython.display import display, Markdown

# Verifikasi GPU
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('GPU tidak tersedia — gunakan CPU (lebih lambat)')


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
/tmp/ipykernel_22/970933085.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS as LangFAISS
/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1432: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4
VRAM: 15.6 GB


## Section 3 — Autentikasi Hugging Face

In [ ]:
# ── Colab Secrets ──
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print('Menggunakan Google Colab Secrets')
except Exception:
    # Fallback: Kaggle Secrets
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    print('Menggunakan Kaggle Secrets')

hf_login(token=hf_token)
print('Login Hugging Face: OK')

HF_USERNAME   = 'antonius-vincent'
HF_MODEL_REPO = f'{HF_USERNAME}/Qwen2.5-3B-Instruct-Indonesian-QLoRA'
print(f'Model target: {HF_MODEL_REPO}')


Menggunakan Kaggle Secrets
Login Hugging Face: OK
Model target: antonius-vincent/Qwen2.5-3B-Instruct-Indonesian-QLoRA


## Section 4 — Load Model Fine-Tuned

Memuat model `antonius-vincent/Qwen2.5-3B-Instruct-Indonesian-QLoRA` menggunakan Unsloth dengan 4-bit quantization.

In [4]:
MAX_SEQ_LENGTH = 2048

print(f'Memuat model fine-tuned: {HF_MODEL_REPO}')
print('Ini mungkin membutuhkan beberapa menit...')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = HF_MODEL_REPO,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = None,      # auto-detect (float16 di T4)
    load_in_4bit    = True,      # hemat VRAM untuk inference
)
FastLanguageModel.for_inference(model)  # aktifkan mode inference
model.eval()

print(f'Model loaded: {model.__class__.__name__}')
print(f'Vocab size  : {tokenizer.vocab_size:,}')
vram_used = torch.cuda.memory_allocated() / 1e9
print(f'VRAM terpakai: {vram_used:.2f} GB')


Memuat model fine-tuned: antonius-vincent/Qwen2.5-3B-Instruct-Indonesian-QLoRA
Ini mungkin membutuhkan beberapa menit...
==((====))==  Unsloth 2026.7.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded: Qwen2ForCausalLM
Vocab size  : 151,643
VRAM terpakai: 2.11 GB


## Section 5 — Upload & Load PDF

Upload keempat file PDF Undang-Undang ke Colab, kemudian simpan path-nya untuk diproses.

In [5]:
import glob
from pathlib import Path

# Di Kaggle, semua dataset yang di-add otomatis masuk ke folder /kaggle/input/
print("Mencari file PDF di Kaggle Input...")

# Memindai seluruh folder input untuk mencari file berakhiran .pdf
PDF_PATHS = glob.glob("/kaggle/input/**/*.pdf", recursive=True)

print(f"\nJumlah PDF ditemukan: {len(PDF_PATHS)}")
for p in PDF_PATHS:
    print(f"  - {Path(p).name}")

# Validasi wajib 4 dokumen
assert len(PDF_PATHS) == 4, (
    f"Dibutuhkan tepat 4 file PDF, namun ditemukan {len(PDF_PATHS)}. "
    "Pastikan dataset sudah tercentang di panel 'Data' sebelah kanan."
)

Mencari file PDF di Kaggle Input...

Jumlah PDF ditemukan: 4
  - PP Nomor 5 Tahun 2021.pdf
  - UU Nomor 6 Tahun 2023.pdf
  - PP Nomor 51 Tahun 2023.pdf
  - PP Nomor 35 Tahun 2021.pdf


## Section 6 — Ekstraksi Teks dengan PyMuPDF

Ekstrak teks dari setiap halaman PDF menggunakan `fitz` (PyMuPDF). Setiap halaman disimpan sebagai dict dengan info halaman dan file.

In [6]:
def extract_pdf(pdf_path: str) -> List[Dict]:
    """
    Ekstrak teks dari setiap halaman PDF menggunakan PyMuPDF.
    Return list of dict: {filename, page, text}
    """
    doc   = fitz.open(pdf_path)
    fname = Path(pdf_path).name
    pages = []
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text('text')
        if text.strip():
            pages.append({
                'filename': fname,
                'page'    : page_num + 1,
                'text'    : text,
            })
    doc.close()
    return pages

# Ekstrak semua PDF
raw_pages: List[Dict] = []
for pdf_path in PDF_PATHS:
    pages = extract_pdf(pdf_path)
    raw_pages.extend(pages)
    print(f'  {Path(pdf_path).name}: {len(pages)} halaman')

print(f'\nTotal halaman diekstrak: {len(raw_pages)}')
print('\nContoh teks (halaman pertama, 300 karakter):')
print(raw_pages[0]['text'][:300])


  PP Nomor 5 Tahun 2021.pdf: 739 halaman
  UU Nomor 6 Tahun 2023.pdf: 1127 halaman
  PP Nomor 51 Tahun 2023.pdf: 27 halaman
  PP Nomor 35 Tahun 2021.pdf: 56 halaman

Total halaman diekstrak: 1949

Contoh teks (halaman pertama, 300 karakter):
 
LEMBARAN NEGARA 
REPUBLIK INDONESIA 
No.15, 2021 
ADMINISTRASI. Perizinan Berusaha Berbasis 
Risiko. Penyelenggaraan. (Penjelasan dalam 
Tambahan 
Lembaran 
Negara 
Republik 
Indonesia Nomor 6617) 
 
PERATURAN PEMERINTAH REPUBLIK INDONESIA 
NOMOR 5 TAHUN 2021 
TENTANG 
PENYELENGGARAAN PERIZINAN BE


## Section 7 — Text Cleaning

Membersihkan teks hasil ekstraksi: hapus whitespace berlebih, karakter non-ASCII yang tidak relevan, dan artefak PDF.

In [7]:
def clean_text(text: str) -> str:
    """Bersihkan teks hasil ekstraksi PDF."""
    # Hapus karakter kontrol kecuali newline
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    # Gabungkan tanda hubung baris (hyphenation)
    text = re.sub(r'-(\n)', '', text)
    # Normalisasi multiple newline → max 2
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Normalisasi multiple spasi → satu spasi
    text = re.sub(r'[ \t]+', ' ', text)
    # Hapus baris yang hanya berisi angka (nomor halaman)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    return text.strip()

# Terapkan cleaning ke semua halaman
for page in raw_pages:
    page['text'] = clean_text(page['text'])

# Hapus halaman yang terlalu pendek (header/footer saja)
MIN_CHARS = 100
before = len(raw_pages)
raw_pages = [p for p in raw_pages if len(p['text']) >= MIN_CHARS]
print(f'Halaman sebelum filter: {before}')
print(f'Halaman setelah filter : {len(raw_pages)}')
print('\nContoh setelah cleaning (300 karakter):')
print(raw_pages[0]['text'][:300])


Halaman sebelum filter: 1949
Halaman setelah filter : 1559

Contoh setelah cleaning (300 karakter):
LEMBARAN NEGARA 
REPUBLIK INDONESIA 
No.15, 2021 
ADMINISTRASI. Perizinan Berusaha Berbasis 
Risiko. Penyelenggaraan. (Penjelasan dalam 
Tambahan 
Lembaran 
Negara 
Republik 
Indonesia Nomor 6617) 
 
PERATURAN PEMERINTAH REPUBLIK INDONESIA 
NOMOR 5 TAHUN 2021 
TENTANG 
PENYELENGGARAAN PERIZINAN BERU


## Section 8 — Parent Chunks

Membagi teks menjadi **Parent Chunks** (besar, ~1000 karakter) yang akan digunakan sebagai konteks penuh untuk LLM.

In [8]:
PARENT_CHUNK_SIZE    = 1000   # karakter
PARENT_CHUNK_OVERLAP = 100    # karakter overlap antar chunk

def make_parent_chunks(pages: List[Dict]) -> List[Dict]:
    """
    Bagi setiap halaman menjadi Parent Chunks (~1000 karakter).
    Setiap chunk menyimpan metadata: filename, page, parent_id.
    """
    chunks = []
    for page in pages:
        text  = page['text']
        start = 0
        while start < len(text):
            end  = start + PARENT_CHUNK_SIZE
            chunk_text = text[start:end]
            if chunk_text.strip():
                chunks.append({
                    'parent_id': f"p_{uuid.uuid4().hex[:8]}",
                    'text'     : chunk_text,
                    'metadata' : {
                        'filename': page['filename'],
                        'page'    : page['page'],
                    }
                })
            start = end - PARENT_CHUNK_OVERLAP
    return chunks

parent_chunks = make_parent_chunks(raw_pages)

# Simpan dalam dict untuk lookup cepat
parent_dict: Dict[str, Dict] = {c['parent_id']: c for c in parent_chunks}

print(f'Chunk size   : {PARENT_CHUNK_SIZE} karakter')
print(f'Chunk overlap: {PARENT_CHUNK_OVERLAP} karakter')
print(f'Total parent chunks: {len(parent_chunks)}')
print(f'\nContoh parent chunk pertama:')
ex = parent_chunks[0]
print(f'  parent_id: {ex["parent_id"]}')
print(f'  filename : {ex["metadata"]["filename"]}')
print(f'  page     : {ex["metadata"]["page"]}')
print(f'  text     : {ex["text"][:200]}...')


Chunk size   : 1000 karakter
Chunk overlap: 100 karakter
Total parent chunks: 2900

Contoh parent chunk pertama:
  parent_id: p_81d1bcee
  filename : PP Nomor 5 Tahun 2021.pdf
  page     : 1
  text     : LEMBARAN NEGARA 
REPUBLIK INDONESIA 
No.15, 2021 
ADMINISTRASI. Perizinan Berusaha Berbasis 
Risiko. Penyelenggaraan. (Penjelasan dalam 
Tambahan 
Lembaran 
Negara 
Republik 
Indonesia Nomor 6617) 
 
...


## Section 9 — Child Chunks

Membagi Parent Chunk menjadi **Child Chunks** (kecil, ~300 karakter) yang digunakan untuk pencarian semantik dan keyword. Setiap child chunk menyimpan referensi ke parent-nya.

In [9]:
CHILD_CHUNK_SIZE    = 300   # karakter
CHILD_CHUNK_OVERLAP = 50    # karakter overlap antar chunk

def make_child_chunks(parent_chunks: List[Dict]) -> List[Dict]:
    """
    Bagi setiap Parent Chunk menjadi Child Chunks (~300 karakter).
    Setiap child chunk menyimpan parent_id untuk mapping balik.
    """
    chunks = []
    for parent in parent_chunks:
        text  = parent['text']
        start = 0
        while start < len(text):
            end        = start + CHILD_CHUNK_SIZE
            chunk_text = text[start:end]
            if chunk_text.strip():
                chunks.append({
                    'child_id' : f"c_{uuid.uuid4().hex[:8]}",
                    'parent_id': parent['parent_id'],
                    'text'     : chunk_text,
                    'metadata' : {**parent['metadata']},  # copy metadata parent
                })
            start = end - CHILD_CHUNK_OVERLAP
    return chunks

child_chunks = make_child_chunks(parent_chunks)

print(f'Chunk size   : {CHILD_CHUNK_SIZE} karakter')
print(f'Chunk overlap: {CHILD_CHUNK_OVERLAP} karakter')
print(f'Total child chunks: {len(child_chunks)}')
print(f'Rata-rata child per parent: {len(child_chunks)/len(parent_chunks):.1f}')
print(f'\nContoh child chunk pertama:')
ex = child_chunks[0]
print(f'  child_id : {ex["child_id"]}')
print(f'  parent_id: {ex["parent_id"]}')
print(f'  text     : {ex["text"][:150]}...')


Chunk size   : 300 karakter
Chunk overlap: 50 karakter
Total child chunks: 9151
Rata-rata child per parent: 3.2

Contoh child chunk pertama:
  child_id : c_9e4dd8ee
  parent_id: p_81d1bcee
  text     : LEMBARAN NEGARA 
REPUBLIK INDONESIA 
No.15, 2021 
ADMINISTRASI. Perizinan Berusaha Berbasis 
Risiko. Penyelenggaraan. (Penjelasan dalam 
Tambahan 
Lem...


## Section 10 — Metadata Enrichment

Menambahkan metadata ke setiap chunk:
- `filename` : nama file PDF
- `page`     : nomor halaman
- `chapter`  : BAB/Pasal jika terdeteksi
- `parent_id`: ID parent chunk (khusus child)

In [10]:
import re

_BAB_RE   = re.compile(r'\bBAB\s+([IVXLC]+|\d+)\b', re.IGNORECASE)
_PASAL_RE = re.compile(r'\bPasal\s+(\d+)\b', re.IGNORECASE)

def detect_chapter(text: str) -> str:
    """Deteksi BAB atau Pasal teratas dari teks."""
    m = _BAB_RE.search(text)
    if m:
        return f'BAB {m.group(1).upper()}'
    m = _PASAL_RE.search(text)
    if m:
        return f'Pasal {m.group(1)}'
    return 'N/A'

def enrich_metadata(chunks: List[Dict]) -> List[Dict]:
    """Tambahkan field 'chapter' ke metadata setiap chunk."""
    for chunk in chunks:
        chunk['metadata']['chapter'] = detect_chapter(chunk['text'])
    return chunks

# Enrich parent dan child
parent_chunks = enrich_metadata(parent_chunks)
child_chunks  = enrich_metadata(child_chunks)

# Update parent_dict setelah enrichment
parent_dict = {c['parent_id']: c for c in parent_chunks}

# Tampilkan distribusi chapter
from collections import Counter
chapter_counts = Counter(c['metadata']['chapter'] for c in child_chunks)
print('Top 10 chapter terdeteksi:')
for chap, count in chapter_counts.most_common(10):
    print(f'  {chap}: {count} chunks')

print('\nContoh metadata child chunk ke-5:')
ex = child_chunks[5]
print(f'  filename : {ex["metadata"]["filename"]}')
print(f'  page     : {ex["metadata"]["page"]}')
print(f'  chapter  : {ex["metadata"]["chapter"]}')
print(f'  parent_id: {ex["parent_id"]}')


Top 10 chapter terdeteksi:
  N/A: 5655 chunks
  Pasal 40: 83 chunks
  Pasal 1: 63 chunks
  Pasal 26: 55 chunks
  Pasal 17: 53 chunks
  Pasal 4: 47 chunks
  Pasal 24: 43 chunks
  Pasal 28: 43 chunks
  Pasal 35: 43 chunks
  Pasal 38: 42 chunks

Contoh metadata child chunk ke-5:
  filename : PP Nomor 5 Tahun 2021.pdf
  page     : 2
  chapter  : BAB I
  parent_id: p_7aae8d73


## Section 11 — Embedding BAAI/bge-m3 + FAISS

Menggunakan model `BAAI/bge-m3` untuk mengubah child chunk menjadi vektor, lalu disimpan ke FAISS index.

In [11]:
# Buat LangChain Documents dari child chunks
child_docs = [
    Document(
        page_content=c['text'],
        metadata={
            **c['metadata'],
            'child_id' : c['child_id'],
            'parent_id': c['parent_id'],
        }
    )
    for c in child_chunks
]

print(f'Total child Documents: {len(child_docs)}')
print('Memuat model embedding BAAI/bge-m3 ...')
print('(Dijalankan di CPU untuk menghemat VRAM bagi LLM)')

embedding_model = HuggingFaceEmbeddings(
    model_name   = 'BAAI/bge-m3',
    model_kwargs = {'device': 'cpu'},          # CPU agar VRAM bebas untuk LLM
    encode_kwargs= {'normalize_embeddings': True, 'batch_size': 16},
)

print('Membuat FAISS index dari child chunks ...')
child_vectorstore = LangFAISS.from_documents(child_docs, embedding_model)
child_faiss_retriever = child_vectorstore.as_retriever(
    search_kwargs={'k': 10}
)

# Simpan FAISS ke disk (opsional, untuk reuse)
child_vectorstore.save_local('./faiss_index')
print(f'FAISS index tersimpan di ./faiss_index')
print(f'Embedding dimension: {embedding_model._client.get_sentence_embedding_dimension()}')


Total child Documents: 9151
Memuat model embedding BAAI/bge-m3 ...
(Dijalankan di CPU untuk menghemat VRAM bagi LLM)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Membuat FAISS index dari child chunks ...
FAISS index tersimpan di ./faiss_index
Embedding dimension: 1024


/tmp/ipykernel_22/3599185408.py:33: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f'Embedding dimension: {embedding_model._client.get_sentence_embedding_dimension()}')


## Section 12 — BM25 Retriever

Membangun BM25 Retriever berbasis keyword dari child chunks sebagai komplemen pencarian semantik.

In [12]:
# BM25 Retriever menggunakan child_docs yang sama dengan FAISS
bm25_retriever = BM25Retriever.from_documents(
    child_docs,
    k=10,
)

# Test BM25
test_q = 'hak lembur pekerja'
bm25_results = bm25_retriever.invoke(test_q)
print(f'BM25 Retriever dibangun dari {len(child_docs)} dokumen')
print(f'\nTest query: "{test_q}"')
print(f'Jumlah hasil BM25: {len(bm25_results)}')
print(f'Top-1 preview: {bm25_results[0].page_content[:150]}...')


BM25 Retriever dibangun dari 9151 dokumen

Test query: "hak lembur pekerja"
Jumlah hasil BM25: 10
Top-1 preview: mberikan perlindungan terhadap seluruh 
harta milik pekerja migran Indonesia untuk 
kepentingan keluarganya; 
n. 
tidak mengurus pemenuhan semua hak p...


## Section 13 — Ensemble Retriever

Menggabungkan FAISS (semantik) dan BM25 (keyword) dengan bobot yang dapat diatur. Mengambil minimal 5 dokumen.

In [13]:
# Bobot retriever — dapat diubah sesuai kebutuhan
FAISS_WEIGHT = 0.6   # lebih berat ke semantik
BM25_WEIGHT  = 0.4   # keyword sebagai pelengkap
TOP_K        = 10    # ambil 10 child docs (>=5 sesuai rubrik)

child_faiss_retriever = child_vectorstore.as_retriever(
    search_kwargs={'k': TOP_K}
)
bm25_retriever.k = TOP_K

ensemble_retriever = EnsembleRetriever(
    retrievers=[child_faiss_retriever, bm25_retriever],
    weights    =[FAISS_WEIGHT, BM25_WEIGHT],
)

# Test ensemble
test_q       = 'upah lembur pekerja'
ens_results  = ensemble_retriever.invoke(test_q)
print(f'Bobot: FAISS={FAISS_WEIGHT}, BM25={BM25_WEIGHT}')
print(f'TOP_K: {TOP_K}')
print(f'\nTest query: "{test_q}"')
print(f'Jumlah hasil ensemble: {len(ens_results)}')
for i, doc in enumerate(ens_results[:3], 1):
    m = doc.metadata
    print(f'\n  [{i}] {m["filename"]} | hal. {m["page"]} | {m.get("chapter","-")}')
    print(f'      {doc.page_content[:120]}...')


Bobot: FAISS=0.6, BM25=0.4
TOP_K: 10

Test query: "upah lembur pekerja"
Jumlah hasil ensemble: 19

  [1] PP Nomor 35 Tahun 2021.pdf | hal. 19 | N/A
      PRESIDEN
REPUELIK INDONESIA
-L9a. untuk ja- kerja lembur pertama sebesar
1,5 (satu koma lima) kali Upah sejam; dan
b. un...

  [2] PP Nomor 35 Tahun 2021.pdf | hal. 3 | N/A
      u untuk 5 (lima) hari kerja
dalam 1 (satu) minggu atau waktu kerja pada hari
istirahat mingguan dan/atau pada hari libur...

  [3] PP Nomor 35 Tahun 2021.pdf | hal. 19 | N/A
      n Upah Kerja Lembur dilaksanakan
sebagai berikut:
1. jam pertama sampai dengan jam ketujuh,
dibayar 2 (dua) kali Upah se...


## Section 14 — Parent-Child Retriever

Child chunks digunakan untuk retrieval, hasilnya dipetakan kembali ke Parent chunk untuk konteks LLM yang lebih lengkap.

In [14]:
def parent_child_retrieve(query: str, top_k_parents: int = 5) -> List[Dict]:
    """
    Parent-Child Retriever:
    1. Gunakan Ensemble Retriever untuk ambil child chunks (semantik + BM25)
    2. Petakan setiap child → parent chunk menggunakan parent_id
    3. Deduplikasi parent berdasarkan parent_id
    4. Return top-K parent chunks sebagai konteks LLM
    """
    # Langkah 1: Retrieve child chunks via ensemble
    child_results = ensemble_retriever.invoke(query)

    # Langkah 2-3: Map ke parent, deduplikasi
    seen_parent_ids = set()
    parent_results  = []
    for child_doc in child_results:
        pid = child_doc.metadata.get('parent_id')
        if pid and pid not in seen_parent_ids:
            seen_parent_ids.add(pid)
            parent = parent_dict.get(pid)
            if parent:
                parent_results.append({
                    'text'    : parent['text'],
                    'metadata': parent['metadata'],
                    'child_preview': child_doc.page_content[:120],
                })
        if len(parent_results) >= top_k_parents:
            break

    return parent_results


# Test Parent-Child Retriever
test_q   = 'Saya staf admin lembur 3 jam, berhak dapat uang lembur?'
p_results = parent_child_retrieve(test_q, top_k_parents=5)
print(f'Query : "{test_q}"')
print(f'Jumlah parent chunks diambil: {len(p_results)}')
for i, r in enumerate(p_results, 1):
    m = r['metadata']
    print(f'\n  [{i}] {m["filename"]} | Hal. {m["page"]} | {m.get("chapter","-")}')
    print(f'      Child match: {r["child_preview"]}...')
    print(f'      Parent text: {r["text"][:200]}...')


Query : "Saya staf admin lembur 3 jam, berhak dapat uang lembur?"
Jumlah parent chunks diambil: 5

  [1] PP Nomor 35 Tahun 2021.pdf | Hal. 3 | N/A
      Child match: a, kesepakatan, atau peraturan
perundang-undangan, termasuk tunjangan bagi
Pekerja/Buruh dan keluarganya atas suatu peke...
      Parent text: PRESIDEN
REPUBLIK INDONESIA
-35
b. usaha-usaha sosial dan usaha-usaha lain yang
mempunyai pengurus dan mempekerjakan orang
lain dengan membayar Upah atau imbalan dalam
bentuk lain.
Serikat Pekerja/Ser...

  [2] PP Nomor 35 Tahun 2021.pdf | Hal. 18 | Pasal 30
      Child match: mbur berkewajiban:
a. membayar Upah Kerja Lembur;
b. memberi kesempatan untuk 
istirahat
secukupnya; dan
c. memberikan m...
      Parent text: PRESiDEN
REPUBLIK INDONESI,A.
-18(21 Perintah dan persetujuan sebagaimana dimaksud
pada ayat (1) dapat dibuat dalam bentuk daftar
Pekerja/Buruh yang bersedia bekerja lembur yang
ditandatangani oleh 
P...

  [3] PP Nomor 35 Tahun 2021.pdf | Hal. 17 | Pasal 28
      Child

## Section 15 — Prompt Template

Template prompt RAG berisi `{context}` dari dokumen dan `{question}` dari pengguna.

In [15]:
SYSTEM_PROMPT = (
    'Kamu adalah asisten AI hukum yang cerdas dan membantu. '
    'Jawab pertanyaan HANYA berdasarkan dokumen konteks yang diberikan. '
    'Jika informasi tidak ada dalam konteks, katakan bahwa Anda tidak '
    'menemukan informasi tersebut dalam dokumen yang tersedia. '
    'Gunakan Bahasa Indonesia yang baik dan benar.'
)

RAG_PROMPT_TEMPLATE = """Konteks dari dokumen Undang-Undang:
---
{context}
---

Pertanyaan: {question}

Berikan jawaban yang jelas dan lengkap berdasarkan konteks di atas:"""

def build_prompt(context: str, question: str) -> str:
    return RAG_PROMPT_TEMPLATE.format(
        context =context,
        question=question,
    )

# Tampilkan contoh template
sample_context  = '[...teks dari dokumen UU...]'
sample_question = 'Apa syarat untuk mendapat uang lembur?'
print('=== Contoh Prompt Template ===')
print(build_prompt(sample_context, sample_question))


=== Contoh Prompt Template ===
Konteks dari dokumen Undang-Undang:
---
[...teks dari dokumen UU...]
---

Pertanyaan: Apa syarat untuk mendapat uang lembur?

Berikan jawaban yang jelas dan lengkap berdasarkan konteks di atas:


## Section 16 — RAG Pipeline & Inference

Rangkaian penuh: retrieve → format context → generate answer.

In [16]:
def generate_answer(prompt: str, max_new_tokens: int = 512) -> str:
    """Generate jawaban menggunakan model fine-tuned."""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens      = max_new_tokens,
            temperature         = 0.3,
            top_p               = 0.9,
            repetition_penalty  = 1.1,
            do_sample           = True,
            pad_token_id        = tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def format_citations(parent_results: List[Dict]) -> str:
    """Format sitasi sumber dari parent chunks."""
    seen   = set()
    cits   = []
    for r in parent_results:
        m   = r['metadata']
        key = (m['filename'], m['page'])
        if key not in seen:
            seen.add(key)
            chap = m.get('chapter', '-')
            cits.append(f"📄 **{m['filename']}**, Halaman {m['page']}, {chap}")
    return '\n'.join(cits)


def rag_pipeline(question: str, top_k: int = 5) -> Tuple[str, str]:
    """
    Pipeline RAG lengkap:
    1. Retrieve parent chunks via Parent-Child Retriever
    2. Format context
    3. Build prompt
    4. Generate answer
    5. Format citations
    Return (answer, citations)
    """
    # 1. Retrieve
    parent_results = parent_child_retrieve(question, top_k_parents=top_k)

    if not parent_results:
        return ('Maaf, tidak ditemukan informasi yang relevan dalam dokumen.', '')

    # 2. Format context (gabungkan teks parent chunks)
    context_parts = []
    for i, r in enumerate(parent_results, 1):
        m    = r['metadata']
        header = f"[Dok {i}: {m['filename']}, Hal. {m['page']}, {m.get('chapter','-')}]"
        context_parts.append(f"{header}\n{r['text']}")
    context = '\n\n'.join(context_parts)

    # 3. Build prompt
    prompt = build_prompt(context, question)

    # 4. Generate
    answer = generate_answer(prompt)

    # 5. Citations
    citations = format_citations(parent_results)

    return answer, citations


# Test pipeline dengan pertanyaan wajib dari rubrik
test_question = 'Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. '\
                'Apakah saya berhak dapat uang lembur?'

print('=== TEST RAG PIPELINE ===')
print(f'Pertanyaan: {test_question}')
print('Generating...')
answer, citations = rag_pipeline(test_question)

print('\n--- JAWABAN ---')
display(Markdown(answer))
print('\n--- SUMBER ---')
display(Markdown(citations))


=== TEST RAG PIPELINE ===
Pertanyaan: Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?
Generating...


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12


--- JAWABAN ---


Ya, Anda berhak mendapatkan upah lembur karena Anda melakukan kerja lembur selama 3 jam pada hari kerja. Berdasarkan Dokumen 1, Upah Kerja Lembur untuk satu kali lembur pertama sebesar 1,5 kali Upah Sejam, dan untuk setiap lembur berikutnya sebesar 2 kali Upah Sejam. Dalam kasus ini, Anda melakukan kerja lembur selama 3 jam, sehingga Anda berhak mendapatkan upah lembur sesuai dengan aturan yang disebutkan.


--- SUMBER ---


📄 **PP Nomor 35 Tahun 2021.pdf**, Halaman 19, N/A
📄 **PP Nomor 35 Tahun 2021.pdf**, Halaman 18, Pasal 30
📄 **PP Nomor 35 Tahun 2021.pdf**, Halaman 3, N/A
📄 **PP Nomor 35 Tahun 2021.pdf**, Halaman 17, Pasal 28

## Section 17 — Sitasi Sumber

Menambahkan sitasi otomatis berisi nama file PDF dan nomor halaman pada setiap jawaban yang dihasilkan.

In [17]:
# Sitasi sudah terintegrasi di rag_pipeline() melalui format_citations().
# Section ini menampilkan contoh output sitasi secara terpisah.

print('=== CONTOH FORMAT SITASI ===')
sample_results = [
    {'metadata': {'filename': 'UU-13-2003-Ketenagakerjaan.pdf', 'page': 47, 'chapter': 'BAB X'}},
    {'metadata': {'filename': 'PP-35-2021-PKWT.pdf',            'page': 12, 'chapter': 'BAB III'}},
    {'metadata': {'filename': 'UU-13-2003-Ketenagakerjaan.pdf', 'page': 48, 'chapter': 'BAB X'}},
]
print(format_citations(sample_results))

print('\nPenjelasan:')
print('  - Setiap sumber ditampilkan satu kali (deduplikasi)')
print('  - Format: nama file | nomor halaman | BAB/Pasal')
print('  - Sitasi otomatis dihasilkan dari metadata chunk yang diambil')


=== CONTOH FORMAT SITASI ===
📄 **UU-13-2003-Ketenagakerjaan.pdf**, Halaman 47, BAB X
📄 **PP-35-2021-PKWT.pdf**, Halaman 12, BAB III
📄 **UU-13-2003-Ketenagakerjaan.pdf**, Halaman 48, BAB X

Penjelasan:
  - Setiap sumber ditampilkan satu kali (deduplikasi)
  - Format: nama file | nomor halaman | BAB/Pasal
  - Sitasi otomatis dihasilkan dari metadata chunk yang diambil


## Section 18 — Instalasi Tambahan (Advanced)

Install `duckduckgo-search` dan `sentence-transformers` (untuk CrossEncoder Reranker).

In [18]:
%%capture
!pip install duckduckgo-search -q
# sentence-transformers sudah terinstal di Section 1
from sentence_transformers import CrossEncoder
print('Library advanced siap.')


## Section 19 — HyDE (Hypothetical Document Embeddings)

**HyDE** (Gao et al., 2022) meningkatkan kualitas retrieval dengan cara:
1. Minta LLM membuat **jawaban hipotetis** (halusinasi terkontrol) untuk query
2. Embed jawaban hipotetis tersebut — hasilnya lebih dekat ke ruang vektor dokumen relevan daripada embedding query mentah
3. Rata-ratakan embedding query + semua jawaban hipotetis
4. Gunakan vektor rata-rata untuk mencari di FAISS

```
Query → LLM → [Hyp-1, Hyp-2] → Embed → Average → FAISS Search
```

In [19]:
def generate_hypothetical_answers(question: str, n: int = 2) -> List[str]:
    """
    Generate N jawaban hipotetis menggunakan LLM fine-tuned.
    Jawaban sengaja 'halusinasi' agar embedding lebih dekat ke dokumen relevan.
    """
    hyde_sys = (
        'Kamu adalah ahli hukum ketenagakerjaan Indonesia. '
        'Buatlah jawaban hipotetis seolah kamu punya akses ke UU yang relevan.'
    )
    hyde_user = (
        f'Pertanyaan: {question}\n\n'
        'Tuliskan jawaban hipotetis yang detail, seolah jawaban tersebut '
        'berasal dari dokumen Undang-Undang resmi Indonesia:'
    )
    answers = []
    for _ in range(n):
        msgs = [
            {'role': 'system', 'content': hyde_sys},
            {'role': 'user',   'content': hyde_user},
        ]
        text   = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens   = 200,
                temperature      = 0.85,
                do_sample        = True,
                pad_token_id     = tokenizer.eos_token_id,
            )
        new_tok = out[0][inputs['input_ids'].shape[1]:]
        answers.append(tokenizer.decode(new_tok, skip_special_tokens=True).strip())
    return answers


def hyde_embed_and_retrieve(question: str, n_hypothetical: int = 2, top_k: int = 20) -> List[Dict]:
    """
    HyDE retrieval:
    1. Buat n_hypothetical jawaban hipotetis
    2. Embed query + semua jawaban hipotetis
    3. Rata-ratakan embedding
    4. Cari di FAISS child_vectorstore
    Return: list child chunk dicts
    """
    hyp_answers = generate_hypothetical_answers(question, n=n_hypothetical)
    print(f'  [HyDE] {n_hypothetical} jawaban hipotetis dibuat')
    for i, a in enumerate(hyp_answers, 1):
        print(f'  [HyDE] Hyp-{i}: {a[:100]}...')

    # Embed semua teks
    all_texts = [question] + hyp_answers
    all_embeddings = np.array(
        embedding_model.embed_documents(all_texts), dtype=np.float32
    )
    avg_embedding = all_embeddings.mean(axis=0, keepdims=True)  # (1, dim)

    # Cari di FAISS
    faiss_index = child_vectorstore.index
    _, indices  = faiss_index.search(avg_embedding, top_k)

    results = []
    for idx in indices[0]:
        if 0 <= idx < len(child_chunks):
            results.append(child_chunks[idx])
    return results, hyp_answers


# ── Demo HyDE ──
test_q = 'Berapa batas jam lembur per minggu untuk pekerja?'
print(f'Query: {test_q}')
print('Generating hypothetical answers...')
hyp = generate_hypothetical_answers(test_q, n=2)
for i, a in enumerate(hyp, 1):
    print(f'\nHyp-{i}: {a[:200]}')


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: Berapa batas jam lembur per minggu untuk pekerja?
Generating hypothetical answers...


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Hyp-1: Sebagai AI, saya tidak memiliki akses langsung ke UU atau kebijakan pemerintah saat ini. Namun, secara umum, dalam banyak negara termasuk Indonesia, ada batasan jam kerja harian dan mingguan yang dite

Hyp-2: Menurut Undang-Undang Nomor 13 tahun 2003 tentang Kerja Sama, atau "UU No. 13 Tahun 2003 tentang Kerja Sama," diatur tentang batas jam lembur per minggu untuk pekerja. Batas jam lembur per minggu yang


## Section 20 — Reranker (BAAI/bge-reranker-base)

**Cross-Encoder Reranker** menilai relevansi setiap pasangan (query, dokumen) secara langsung — jauh lebih akurat dari cosine similarity.

Pipeline:
```
Candidates (banyak) → CrossEncoder(query, doc_i) → Score → Sort → Top-3
```

Threshold = **0.35**: jika skor Top-1 di bawah threshold → fallback ke DuckDuckGo.

In [20]:
from sentence_transformers import CrossEncoder

RERANKER_MODEL = 'BAAI/bge-reranker-base'
RERANKER_THRESHOLD = 0.35   # skor minimum untuk pakai dokumen lokal
TOP_K_RERANK = 3            # ambil top-3 setelah reranking

print(f'Memuat reranker: {RERANKER_MODEL}')
reranker = CrossEncoder(
    RERANKER_MODEL,
    device='cpu',   # CPU untuk hemat VRAM bagi LLM
    max_length=512,
)
print(f'Reranker loaded | Threshold: {RERANKER_THRESHOLD} | Top-K: {TOP_K_RERANK}')


def rerank_candidates(
    query: str,
    candidates: List[Dict],
    top_k: int = TOP_K_RERANK,
) -> List[Tuple[Dict, float]]:
    """
    Rerank candidate parent chunks menggunakan CrossEncoder.
    Return list of (chunk_dict, score) terurut descending.
    """
    if not candidates:
        return []
    pairs  = [(query, c['text'][:512]) for c in candidates]
    scores = reranker.predict(pairs, show_progress_bar=False).tolist()
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]


# ── Demo Reranker ──
test_q     = 'hak upah lembur pekerja admin'
test_cands = parent_child_retrieve(test_q, top_k_parents=8)
ranked     = rerank_candidates(test_q, test_cands)
print(f'\nQuery: "{test_q}"')
print(f'Candidates: {len(test_cands)} | Setelah rerank: {len(ranked)}')
for i, (doc, score) in enumerate(ranked, 1):
    m = doc['metadata']
    print(f'  [{i}] Score={score:.4f} | {m["filename"]} | Hal.{m["page"]} | {m.get("chapter","-")}')
print(f'\nTop-1 score: {ranked[0][1]:.4f} (threshold={RERANKER_THRESHOLD})')
print('Keputusan :', 'LOKAL ✅' if ranked[0][1] >= RERANKER_THRESHOLD else 'FALLBACK 🌐')


Memuat reranker: BAAI/bge-reranker-base


config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

Reranker loaded | Threshold: 0.35 | Top-K: 3

Query: "hak upah lembur pekerja admin"
Candidates: 8 | Setelah rerank: 3
  [1] Score=0.4037 | PP Nomor 35 Tahun 2021.pdf | Hal.17 | Pasal 28
  [2] Score=0.3139 | PP Nomor 35 Tahun 2021.pdf | Hal.3 | N/A
  [3] Score=0.1404 | UU Nomor 6 Tahun 2023.pdf | Hal.559 | Pasal 79

Top-1 score: 0.4037 (threshold=0.35)
Keputusan : LOKAL ✅


## Section 21 — Advanced RAG Pipeline

Pipeline lengkap Advanced:

```
Query
  │
  ├─► HyDE: buat 2 jawaban hipotetis → embed → rata-ratakan → FAISS
  │
  ├─► Ensemble Retriever (FAISS + BM25)
  │
  ├─► Gabungkan & Map ke Parent Chunks
  │
  ├─► Reranker (BAAI/bge-reranker-base) → Top-3
  │
  └─► Cek Relevance Score Top-1
        ├─ score >= 0.35 → Generate dari dokumen lokal
        └─ score <  0.35 → DuckDuckGo Fallback → Generate dari web
```

In [21]:
from duckduckgo_search import DDGS

def duckduckgo_fallback(query: str, max_results: int = 3) -> str:
    """Cari informasi dari internet menggunakan DuckDuckGo."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(
                f'{query} undang-undang Indonesia ketenagakerjaan',
                max_results=max_results,
            ))
        if not results:
            return 'Tidak ditemukan hasil pencarian internet yang relevan.'
        parts = []
        for r in results:
            parts.append(f"[{r.get('title','-')}]\n{r.get('body','')}\nURL: {r.get('href','')}")
        return '\n\n'.join(parts)
    except Exception as e:
        return f'DuckDuckGo search error: {e}'


def advanced_rag_pipeline(
    question: str,
    n_hypothetical: int = 2,
    top_k_rerank: int = TOP_K_RERANK,
    threshold: float = RERANKER_THRESHOLD,
) -> Tuple[str, str, str]:
    """
    Advanced RAG Pipeline (HyDE + Reranker + DuckDuckGo Fallback).
    Return: (answer, citations, pipeline_log)
    """
    log = []

    # ── Step 1: HyDE retrieval ──
    log.append('Step 1: HyDE retrieval')
    hyde_children, hyp_answers = hyde_embed_and_retrieve(
        question, n_hypothetical=n_hypothetical, top_k=20
    )

    # ── Step 2: Ensemble retrieval (query mentah) ──
    log.append('Step 2: Ensemble retrieval')
    ensemble_docs = ensemble_retriever.invoke(question)

    # ── Step 3: Gabungkan & map ke parent ──
    log.append('Step 3: Map child → parent')
    seen_pids  = set()
    candidates = []

    all_children = hyde_children + [
        {'parent_id': d.metadata.get('parent_id'), 'text': d.page_content, 'metadata': d.metadata}
        for d in ensemble_docs
    ]
    for c in all_children:
        pid = c.get('parent_id')
        if pid and pid not in seen_pids:
            seen_pids.add(pid)
            parent = parent_dict.get(pid)
            if parent:
                candidates.append(parent)
    log.append(f'  Kandidat parent: {len(candidates)}')

    # ── Step 4: Rerank ──
    log.append('Step 4: Reranking')
    ranked = rerank_candidates(question, candidates, top_k=top_k_rerank)
    if not ranked:
        return ('Tidak ada dokumen relevan ditemukan.', '', '\n'.join(log))

    top1_score = ranked[0][1]
    log.append(f'  Top-1 reranker score: {top1_score:.4f} (threshold={threshold})')

    # ── Step 5-8: Branch ──
    if top1_score >= threshold:
        log.append('Step 5: LOKAL — score >= threshold, gunakan dokumen lokal')
        top_parents = [doc for doc, _ in ranked]

        context_parts = []
        for i, p in enumerate(top_parents, 1):
            m = p['metadata']
            context_parts.append(
                f"[Dok {i}: {m['filename']}, Hal.{m['page']}, {m.get('chapter','-')}]\n{p['text']}"
            )
        context  = '\n\n'.join(context_parts)
        prompt   = build_prompt(context, question)
        answer   = generate_answer(prompt)
        citations = format_citations([{'metadata': p['metadata']} for p in top_parents])
        citations = f'*Sumber: dokumen lokal*\n{citations}'

    else:
        log.append('Step 5: FALLBACK — score < threshold, gunakan DuckDuckGo')
        web_context = duckduckgo_fallback(question)
        prompt      = build_prompt(web_context, question)
        answer      = generate_answer(prompt)
        citations   = '*Sumber: DuckDuckGo Web Search (dokumen lokal tidak cukup relevan)*'

    return answer, citations, '\n'.join(log)


# ── Test wajib rubrik ──
WAJIB_Q = (
    'Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. '
    'Apakah saya berhak dapat uang lembur?'
)
print('=== TEST ADVANCED RAG PIPELINE ===')
print(f'Pertanyaan: {WAJIB_Q}')
print()
answer_adv, cit_adv, log_adv = advanced_rag_pipeline(WAJIB_Q)
print('--- Pipeline Log ---')
print(log_adv)
print('\n--- Jawaban ---')
display(Markdown(answer_adv))
print('\n--- Sumber ---')
display(Markdown(cit_adv))


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== TEST ADVANCED RAG PIPELINE ===
Pertanyaan: Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?



Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [HyDE] 2 jawaban hipotetis dibuat
  [HyDE] Hyp-1: Menurut UU Nomor 13 tahun 2003 tentang Kepegawaian, atau Undang-Undang Kepegawaian, uang lembur dibe...
  [HyDE] Hyp-2: Dalam hal ini, Anda memiliki hak untuk menerima gaji tambahan karena bekerja di luar jam kerja biasa...


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Pipeline Log ---
Step 1: HyDE retrieval
Step 2: Ensemble retrieval
Step 3: Map child → parent
  Kandidat parent: 16
Step 4: Reranking
  Top-1 reranker score: 0.6539 (threshold=0.35)
Step 5: LOKAL — score >= threshold, gunakan dokumen lokal

--- Jawaban ---


Menurut dokumen yang disediakan, jika pengusaha memiliki lebih dari 20 orang pekerja/buruh, mereka wajib membayar upah kerja lembur. Namun, ketentuan waktu kerja lembur dan upah kerja lembur diatur dalam Peraturan Pemerintah. Oleh karena itu, untuk mengetahui apakah Anda berhak mendapatkan upah kerja lembur, Anda harus memeriksa kebijakan pengusaha tentang waktu kerja lembur dan upah kerja lembur. Selain itu, waktu kerja lembur hanya boleh dilaksanakan jika ada perintah dari pengusaha dan persetujuan dari pekerja/buruh yang bersangkutan secara tertulis dan/atau melalui media digital.


--- Sumber ---


*Sumber: dokumen lokal*
📄 **PP Nomor 35 Tahun 2021.pdf**, Halaman 3, N/A
📄 **PP Nomor 35 Tahun 2021.pdf**, Halaman 17, Pasal 28
📄 **UU Nomor 6 Tahun 2023.pdf**, Halaman 559, Pasal 79

## Section 22 — Interface Gradio

Antarmuka sederhana: satu input teks (pertanyaan) dan satu output teks (jawaban + sitasi).

In [22]:
def gradio_advanced_rag(question: str) -> str:
    if isinstance(question, list):
        question = question[0] if question else ''
    
    if not question.strip():
        return 'Silakan masukkan pertanyaan terlebih dahulu.'
    answer, citations, log = advanced_rag_pipeline(question)
    output = answer
    if citations:
        output += f'\n\n---\n**Sumber Referensi:**\n{citations}'
    output += f'\n\n---\n**Pipeline Info:**\n```\n{log}\n```'
    return output

EXAMPLE_QUESTIONS = [
    ['Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?'],
    ['Apakah PKWT (kontrak kerja waktu tertentu) boleh mensyaratkan masa percobaan kerja?'],
    ['Apa saja hak cuti yang dimiliki pekerja berdasarkan UU Ketenagakerjaan?'],
    ['Bagaimana ketentuan upah minimum menurut peraturan yang berlaku?'],
    ['Apa yang dimaksud dengan PHK dan apa saja alasan yang diperbolehkan?'],   
]

demo = gr.Interface(
    fn          = gradio_advanced_rag,
    inputs      = gr.Textbox(
        label      = 'Pertanyaan Hukum',
        placeholder= 'Ketik pertanyaan tentang ketenagakerjaan...',
        lines      = 3,
    ),
    outputs     = gr.Textbox(
        label= 'Jawaban & Referensi',
        lines= 18,
    ),
    title       = '⚖️ Asisten Hukum Ketenagakerjaan Indonesia (Advanced RAG)',
    description = (
        'Menggunakan HyDE + Ensemble Retriever + Reranker + DuckDuckGo Fallback. '
        'Jawaban dihasilkan dari dokumen UU resmi Indonesia.'
    ),
    examples    = EXAMPLE_QUESTIONS,
    theme       = gr.themes.Soft(),
)

demo.launch(share=True, debug=False, prevent_thread_lock=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://28e90ad07dff9660ce.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Section 23 — Contoh Pertanyaan & Jawaban

Menguji pipeline RAG dengan beberapa pertanyaan terkait Undang-Undang ketenagakerjaan Indonesia.

In [23]:
# Jalankan beberapa pertanyaan contoh dan tampilkan hasilnya

test_cases = [
    ['Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?'],
    ['Apakah PKWT (kontrak kerja waktu tertentu) boleh mensyaratkan masa percobaan kerja?'],
    ['Apa saja hak cuti yang dimiliki pekerja berdasarkan UU Ketenagakerjaan?'],
    ['Bagaimana ketentuan upah minimum menurut peraturan yang berlaku?'],
    ['Apa yang dimaksud dengan PHK dan apa saja alasan yang diperbolehkan?'],
]

print('=== CONTOH PERTANYAAN & JAWABAN ===')
for i, q in enumerate(test_cases, 1):
    if isinstance(q, list):        
        q = q[0]
    print(f'\n{"="*60}')
    print(f'Pertanyaan {i}: {q}')
    print('='*60)
    answer, citations = rag_pipeline(q)
    display(Markdown(f'**Jawaban:**\n{answer}'))
    if citations:
        display(Markdown(f'**Sumber:**\n{citations}'))
    print()


=== CONTOH PERTANYAAN & JAWABAN ===

Pertanyaan 1: Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


**Jawaban:**
Ya, Anda berhak mendapatkan upah lembur karena Anda melakukan kerja lembur selama 3 jam pada hari kerja biasa. Menurut undang-undang, Anda memiliki hak untuk mengambil upah lembur sebesar 1,5 kali upah sejam atau dua kali upah sejam untuk setiap jam kerja lembur berikutnya. Namun, perlu dicatat bahwa perhitungan upah lembur dapat bervariasi tergantung pada jenis pekerjaan dan aturan yang berlaku di tempat kerja Anda. Selain itu, upah lembur hanya dapat dibayarkan jika kerja lembur dilakukan pada hari istirahat mingguan dan/atau hari libur resmi untuk waktu kerja 5 hari kerja dan 40 jam seminggu.

**Sumber:**
📄 **PP Nomor 35 Tahun 2021.pdf**, Halaman 19, N/A
📄 **PP Nomor 35 Tahun 2021.pdf**, Halaman 18, Pasal 30
📄 **PP Nomor 35 Tahun 2021.pdf**, Halaman 3, N/A
📄 **PP Nomor 35 Tahun 2021.pdf**, Halaman 17, Pasal 28



Pertanyaan 2: Apakah PKWT (kontrak kerja waktu tertentu) boleh mensyaratkan masa percobaan kerja?


...

**Sumber:**
📄 **UU Nomor 6 Tahun 2023.pdf**, Halaman 579, Pasal 82
📄 **PP Nomor 51 Tahun 2023.pdf**, Halaman 2, Pasal 24
📄 **UU Nomor 6 Tahun 2023.pdf**, Halaman 563, N/A
📄 **UU Nomor 6 Tahun 2023.pdf**, Halaman 564, Pasal 91
📄 **PP Nomor 51 Tahun 2023.pdf**, Halaman 17, N/A



Pertanyaan 5: Apa yang dimaksud dengan PHK dan apa saja alasan yang diperbolehkan?


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Unsloth: Input IDs of shape torch.Size([1, 2103]) with length 2103 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if you correct this issue first.


**Jawaban:**
aturan
Pasal 40 ayat (21;
d. uang penggantian hak sesuai ketentuan Pasal 40
ayat (4).
Pasal 40
Pengusaha dapat melakukan Pemutusan Hubungan Kerja
terhadap Pekerja/Buruh karena alasan:
a. Pekerja/Buruh memasuki usia pensiun maka Pekerja/
Buruh berhak atas:
uang pesangon sebesar 1,75 (satu koma tujuh
lima) kali ketentuan Pasal 40 ayat (2);
b. uang penghargaan masa kerja sebesar 1 (satu)
kali ketentuan Pasal 40 ayat (3); dan
c. uang penggantian hak sesuai ketentuan Pasal 40
ayat (4).

**Sumber:**
📄 **PP Nomor 35 Tahun 2021.pdf**, Halaman 55, Pasal 54
📄 **UU Nomor 6 Tahun 2023.pdf**, Halaman 569, Pasal 154
📄 **PP Nomor 35 Tahun 2021.pdf**, Halaman 36, Pasal 40
📄 **PP Nomor 35 Tahun 2021.pdf**, Halaman 27, Pasal 41
📄 **PP Nomor 35 Tahun 2021.pdf**, Halaman 33, Pasal 40